In [ ]:
# Qwen-ATLAS: Architecture Evaluation Pipeline. This notebook serves as the validation and benchmarking pipeline for **Qwen-ATLAS**. It evaluates the baseline performance of `Qwen2.5-7B-Instruct` against a custom-built, routed **Retrieval-Augmented Generation (RAG)** matrix mapped directly to the **MITRE ATT&CK Framework**.

## 1. Environment Setup & Workspace Initialization

In [ ]:
# Install required core packages silently
!pip install -q \
    chromadb==1.5.9 \
    sentence-transformers \
    mitreattack-python \
    transformers \
    accelerate \
    peft \
    bitsandbytes \
    numpy==1.26.4

import sys
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# Map workspace datasets and stage target structural intelligence files
sys.path.append("/kaggle/input/datasets/karthiksethuraman6/qwen-atlas-data")
!cp /kaggle/input/datasets/karthiksethuraman6/qwen-atlas-data/enterprise-attack.json .
!cp /kaggle/input/datasets/karthiksethuraman6/qwen-atlas-data/index_mappings.json .

import chroma_rag
print("Workspace setup and dependencies successfully initialized.")

In [ ]:
## 2. Vector Database & Knowledge Base Ingestion. Populate the vectorized instance of the MITRE ATT&CK enterprise dataset into the local ChromaDB storage layers via structured ingestion routines.

In [ ]:
print("Populating ChromaDB collections...")
chroma_rag.ingest_techniques()
chroma_rag.ingest_groups()
chroma_rag.ingest_mitigations()
print("Knowledge base initialization complete.")

In [ ]:
## 3. Large Language Model Initialization. Load the baseline `Qwen2.5-7B-Instruct` model weights utilizing FP16 precision and automated device orchestration layouts.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()
print("Qwen 2.5 7B loaded in evaluation state.")

In [ ]:
## 4. Evaluation Suite Phase 1: Zero-Shot Base Model Performance. Establish a control baseline by putting raw analytical prompts straight to the model without external contextual enhancement or structured guardrails.

In [ ]:
def generate_base_response(prompt, max_new_tokens=512):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None
        )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

BASE_QUERIES = [
    "What is T1003?", "What is T1055?", "What is T1195.002?", "What is T1110.003?", "What is T1021.001?",
    "What is Process Injection?", "What is OS Credential Dumping?", "Explain Password Spraying.", "What is PowerShell?", "What is Windows Command Shell?",
    "What is APT29?", "What is APT33?", "What is Lazarus Group?", "What is FIN7?", "What is Kimsuky?",
    "What techniques does APT29 use?", "What techniques does Cozy Bear use?", "What techniques does The Dukes use?", "What techniques does Lazarus Group use?",
    "Which groups use T1003?", "Who uses T1055?", "Which groups use T1110.003?", "Which groups use T1195.002?", "Which groups use T1021.001?",
    "How can T1003 be mitigated?", "How can T1055 be mitigated?", "What mitigations exist for T1110.003?", "How can Password Spraying be mitigated?", "How can Process Injection be mitigated?",
    "What credential access techniques does APT29 use?", "What persistence techniques does APT29 use?", "What discovery techniques does Lazarus Group use?", "What lateral movement techniques does FIN7 use?", "What execution techniques does APT33 use?",
    "How do attackers dump credentials?", "Explain credential dumping to a SOC analyst.", "Why is password spraying dangerous?", "What should defenders monitor for Process Injection?", "What indicators suggest PowerShell abuse?", "What attack chain would APT29 likely use after initial access?"
]

base_results = []
for q in BASE_QUERIES:
    print(f"Processing Query: {q}")
    response = generate_base_response(q)
    base_results.append({"query": q, "response": response})

df_base = pd.DataFrame(base_results)
df_base.to_csv("base_7b_results.csv", index=False)
print(f"Phase 1 evaluation finalized. Saved {len(df_base)} control samples to 'base_7b_results.csv'.")

In [ ]:
## 5. Evaluation Suite Phase 2: Hybrid Routed RAG Performance. Execute the identical query suite utilizing the finalized dynamic `smart_retrieve` router to extract context from ChromaDB before injecting structured prompts into the model layer.

In [ ]:
def generate_rag_response(query, max_new_tokens=512):
    context, router = chroma_rag.retrieve(query)
    
    prompt = f"""You are Qwen-ATLAS, a cybersecurity threat intelligence assistant specializing in the MITRE ATT&CK framework.

Guidelines:
- Use the provided context when available.
- Prefer factual and verifiable information.
- If the context is insufficient, use general cybersecurity knowledge.
- If uncertain, clearly indicate uncertainty.
- Do not fabricate ATT&CK techniques, IDs, groups, mitigations, or relationships.
- Keep responses concise, structured, and analyst-oriented.
- When available, include ATT&CK IDs, technique names, group names, and mitigation IDs.
- Do not mention retrieval systems, vector databases, prompts, or internal reasoning.

Context:
{context}

Question:
{query}

Answer:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
    )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return router, response.strip()

CATEGORIZED_QUERIES = [
    {"id": "A1", "query": "What is T1003?", "category": "Technique ID Lookup"},
    {"id": "A2", "query": "What is T1055?", "category": "Technique ID Lookup"},
    {"id": "A3", "query": "What is T1195.002?", "category": "Technique ID Lookup"},
    {"id": "A4", "query": "What is T1110.003?", "category": "Technique ID Lookup"},
    {"id": "A5", "query": "What is T1021.001?", "category": "Technique ID Lookup"},
    {"id": "B1", "query": "What is Process Injection?", "category": "Technique Name Lookup"},
    {"id": "B2", "query": "What is OS Credential Dumping?", "category": "Technique Name Lookup"},
    {"id": "B3", "query": "Explain Password Spraying.", "category": "Technique Name Lookup"},
    {"id": "B4", "query": "What is PowerShell?", "category": "Technique Name Lookup"},
    {"id": "B5", "query": "What is Windows Command Shell?", "category": "Technique Name Lookup"},
    {"id": "C1", "query": "What is APT29?", "category": "Group Information"},
    {"id": "C2", "query": "What is APT33?", "category": "Group Information"},
    {"id": "C3", "query": "What is Lazarus Group?", "category": "Group Information"},
    {"id": "C4", "query": "What is FIN7?", "category": "Group Information"},
    {"id": "C5", "query": "What is Kimsuky?", "category": "Group Information"},
    {"id": "D1", "query": "What techniques does APT29 use?", "category": "Group to Techniques"},
    {"id": "D2", "query": "What techniques does Cozy Bear use?", "category": "Group to Techniques"},
    {"id": "D3", "query": "What techniques does The Dukes use?", "category": "Group to Techniques"},
    {"id": "D4", "query": "What techniques does Lazarus Group use?", "category": "Group to Techniques"},
    {"id": "E1", "query": "Which groups use T1003?", "category": "Technique to Groups"},
    {"id": "E2", "query": "Who uses T1055?", "category": "Technique to Groups"},
    {"id": "E3", "query": "Which groups use T1110.003?", "category": "Technique to Groups"},
    {"id": "E4", "query": "Which groups use T1195.002?", "category": "Technique to Groups"},
    {"id": "E5", "query": "Which groups use T1021.001?", "category": "Technique to Groups"},
    {"id": "F1", "query": "How can T1003 be mitigated?", "category": "Mitigation Lookup"},
    {"id": "F2", "query": "How can T1055 be mitigated?", "category": "Mitigation Lookup"},
    {"id": "F3", "query": "What mitigations exist for T1110.003?", "category": "Mitigation Lookup"},
    {"id": "F4", "query": "How can Password Spraying be mitigated?", "category": "Mitigation Lookup"},
    {"id": "F5", "query": "How can Process Injection be mitigated?", "category": "Mitigation Lookup"},
    {"id": "G1", "query": "What credential access techniques does APT29 use?", "category": "Tactic-Filtered Group Query"},
    {"id": "G2", "query": "What persistence techniques does APT29 use?", "category": "Tactic-Filtered Group Query"},
    {"id": "G3", "query": "What discovery techniques does Lazarus Group use?", "category": "Tactic-Filtered Group Query"},
    {"id": "G4", "query": "What lateral movement techniques does FIN7 use?", "category": "Tactic-Filtered Group Query"},
    {"id": "G5", "query": "What execution techniques does APT33 use?", "category": "Tactic-Filtered Group Query"},
    {"id": "H1", "query": "How do attackers dump credentials?", "category": "Analyst Semantic Query"},
    {"id": "H2", "query": "Explain credential dumping to a SOC analyst.", "category": "Analyst Semantic Query"},
    {"id": "H3", "query": "Why is password spraying dangerous?", "category": "Analyst Semantic Query"},
    {"id": "H4", "query": "What should defenders monitor for Process Injection?", "category": "Analyst Semantic Query"},
    {"id": "H5", "query": "What indicators suggest PowerShell abuse?", "category": "Analyst Semantic Query"},
    {"id": "H6", "query": "What attack chain would APT29 likely use after initial access?", "category": "Analyst Semantic Query"}
]

rag_results = []
for item in CATEGORIZED_QUERIES:
    print(f"Routing Pipeline Triggered [{item['id']}]: {item['query']}")
    router, response = generate_rag_response(item["query"])
    
    rag_results.append({
        "id": item["id"],
        "category": item["category"],
        "query": item["query"],
        "router": router,
        "response": response
    })

df_rag = pd.DataFrame(rag_results)
df_rag.to_csv("base_7b_rag_results.csv", index=False)
print(f"Phase 2 evaluation finalized. Saved {len(df_rag)} routed context samples to 'base_7b_rag_results.csv'.")